# TP Final - Herramientas para Grandes Volúmenes de Datos
## Predicción de tarifa (`fare_amount`) - NYC Yellow Taxi

### Descripción del dataset y su adecuación al curso
Muestra aleatoria (~400.000 filas) del dataset público **NYC Yellow Taxi Trip Records**
(NYC Taxi & Limousine Commission), tomada de 4 meses distintos (2015-01, 2016-01, 2016-02, 2016-03)
del dataset completo (~47 millones de filas originales, ~7.2 GB). El dataset completo es de escala
big data; se trabajó sobre una muestra representativa para hacer viable la experimentación dentro
del curso, procesando la limpieza y el feature engineering con PySpark antes de reducir a un
subconjunto manejable para el entrenamiento con scikit-learn.

**Objetivo de negocio:** estimar la tarifa de un viaje de taxi (`fare_amount`) a partir de datos
conocidos antes/durante el viaje (distancia, ubicación, hora, pasajeros) — el mismo problema que
resuelven apps como Uber/Cabify al mostrar un precio estimado antes de confirmar el viaje.

In [ ]:
%pip install --upgrade pydantic optuna shap
%pip uninstall -y evidently
%pip install evidently

In [ ]:
dbutils.library.restartPython()

### Configuración del proyecto

In [ ]:
# Semilla fija para reproducibilidad (muestreo, split, Random Forest, Optuna)
SEED = 584154

# Tabla Delta con la muestra de NYC Taxi, registrada en Unity Catalog (Databricks Free Edition)
TABLE_NAME = "workspace.default.nyctaxi_sample"

# Experimento de MLflow donde quedan registradas las corridas
MLFLOW_EXPERIMENT = "/Users/rsanchezfaris@itba.edu.ar/nuevoexperimento"

# Nombre del modelo en MLflow Model Registry (Unity Catalog)
MODEL_NAME = "workspace.default.nyctaxi_fare_model"

## 1. Carga y exploración con PySpark

In [ ]:
df = spark.table(TABLE_NAME)
print("Filas:", df.count())
df.printSchema()
display(df.limit(10))

## 2. Limpieza y feature engineering con PySpark

Se filtran valores inválidos (tarifas/distancias negativas o extremas, coordenadas fuera de NYC,
duraciones absurdas) y se generan features nuevas: distancia geodésica (Haversine) a partir de
lat/long real, duración del viaje, y componentes de fecha/hora.

**Nota:** en un escenario de producción real, `trip_distance` y la duración se conocerían solo como
*estimación* (via un motor de ruteo tipo Google Maps/OSRM) antes de iniciar el viaje, no como el
valor final del taxímetro. Para este TP se usan los valores reales como proxy de esa estimación.

In [ ]:
from pyspark.sql import functions as F

R_KM = 6371.0

df_clean = (
    df
    .withColumn("tpep_pickup_datetime", F.to_timestamp("tpep_pickup_datetime"))
    .withColumn("tpep_dropoff_datetime", F.to_timestamp("tpep_dropoff_datetime"))
    .filter(F.col("fare_amount") > 0)
    .filter(F.col("fare_amount") < 250)
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("trip_distance") < 100)
    .filter(F.col("passenger_count").between(1, 6))
    .filter(F.col("pickup_latitude").between(40.4, 41.0))
    .filter(F.col("pickup_longitude").between(-74.3, -73.6))
    .filter(F.col("dropoff_latitude").between(40.4, 41.0))
    .filter(F.col("dropoff_longitude").between(-74.3, -73.6))
)

df_feat = (
    df_clean
    .withColumn(
        "haversine_km",
        F.lit(2) * F.lit(R_KM) * F.asin(
            F.sqrt(
                F.pow(F.sin((F.radians(F.col("dropoff_latitude")) - F.radians(F.col("pickup_latitude"))) / 2), 2)
                + F.cos(F.radians(F.col("pickup_latitude"))) * F.cos(F.radians(F.col("dropoff_latitude")))
                * F.pow(F.sin((F.radians(F.col("dropoff_longitude")) - F.radians(F.col("pickup_longitude"))) / 2), 2)
            )
        )
    )
    .withColumn(
        "trip_duration_min",
        (F.col("tpep_dropoff_datetime").cast("long") - F.col("tpep_pickup_datetime").cast("long")) / 60.0
    )
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("pickup_dayofweek", F.dayofweek("tpep_pickup_datetime"))
    .withColumn("pickup_month", F.month("tpep_pickup_datetime"))
    .withColumn("pickup_year", F.year("tpep_pickup_datetime"))
    .filter(F.col("trip_duration_min").between(1, 180))
    .withColumn(
        "period",
        F.when(F.col("pickup_year") == 2015, F.lit("reference")).otherwise(F.lit("current"))
    )
)

print("Filas luego de limpiar:", df_feat.count())
display(df_feat.groupBy("period").count())

In [ ]:
FEATURE_COLS = [
    "trip_distance", "haversine_km", "passenger_count",
    "pickup_hour", "pickup_dayofweek", "pickup_month",
    "RateCodeID", "payment_type", "VendorID", "trip_duration_min",
]
TARGET_COL = "fare_amount"

model_df = df_feat.select(*FEATURE_COLS, TARGET_COL, "period")

## 3. Muestra de trabajo para scikit-learn
El procesamiento pesado (limpieza, features) ya se hizo en Spark sobre todo el dataset.
Para entrenar con sklearn/Optuna en tiempos razonables, se toma una submuestra manejable.

In [ ]:
from sklearn.model_selection import train_test_split

work_pd = model_df.sample(fraction=0.2, seed=SEED).toPandas()
print("Filas para entrenamiento:", len(work_pd))

X = work_pd[FEATURE_COLS]
y = work_pd[TARGET_COL]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

## 4. Experimentos con MLflow Tracking
Al menos 4 corridas variando hiperparámetros de `RandomForestRegressor`.

In [ ]:
import mlflow
import mlflow.sklearn
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from mlflow.models.signature import infer_signature

mlflow.set_experiment(MLFLOW_EXPERIMENT)

param_grid = [
    {"n_estimators": 50, "max_depth": 5},
    {"n_estimators": 100, "max_depth": 8},
    {"n_estimators": 150, "max_depth": 12},
    {"n_estimators": 200, "max_depth": None},
]

best_rmse = float("inf")
best_run_id = None

for params in param_grid:
    with mlflow.start_run(run_name=f"rf_{params['n_estimators']}_{params['max_depth']}") as run:
        model = RandomForestRegressor(random_state=SEED, n_jobs=-1, **params)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
        mae = float(mean_absolute_error(y_test, y_pred))
        r2 = float(r2_score(y_test, y_pred))

        signature = infer_signature(X_test, y_pred)

        mlflow.log_params(params)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("r2", r2)
        mlflow.sklearn.log_model(model, name="model", signature=signature)

        if rmse < best_rmse:
            best_rmse = rmse
            best_run_id = run.info.run_id

print("Mejor run (grid manual):", best_run_id, "RMSE:", best_rmse)

## 5. Optimización de hiperparámetros con Optuna (opcional)

In [ ]:
import optuna

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300, step=50),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
    }
    model = RandomForestRegressor(random_state=SEED, n_jobs=-1, **params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    trial.set_user_attr("model", model)
    trial.set_user_attr("y_pred", y_pred)
    return rmse

study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=15)

top_trials = sorted(study.trials, key=lambda t: t.value)[:3]

for trial in top_trials:
    model = trial.user_attrs["model"]
    y_pred = trial.user_attrs["y_pred"]
    rmse = trial.value
    mae = float(mean_absolute_error(y_test, y_pred))
    r2 = float(r2_score(y_test, y_pred))
    signature = infer_signature(X_test, y_pred)
    with mlflow.start_run(run_name=f"optuna_trial_{trial.number}") as run:
        mlflow.log_params(trial.params)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("r2", r2)
        mlflow.sklearn.log_model(model, name="model", signature=signature)
        if rmse < best_rmse:
            best_rmse = rmse
            best_run_id = run.info.run_id

print("Mejor run tras Optuna:", best_run_id, "RMSE:", best_rmse)

## 6. Métricas comparativas de todas las corridas

In [ ]:
runs_df = mlflow.search_runs(order_by=["metrics.rmse ASC"])
display(runs_df[["run_id", "params.n_estimators", "params.max_depth", "metrics.rmse", "metrics.mae", "metrics.r2"]])

### Justificación del modelo final

Se eligió el run `rf_150_12` (`RandomForestRegressor` con `n_estimators=150`, `max_depth=12`)
porque presentó el menor RMSE (0.91 USD) y MAE (0.29 USD) de las 7 corridas registradas (4 del
grid manual + 3 mejores de Optuna), con R²=0.991.

La configuración sin límite de profundidad (`max_depth=None`, RMSE=0.9111) obtuvo un resultado
prácticamente idéntico, pero con árboles más profundos y mayor costo de entrenamiento/inferencia
sin ganancia real, por lo que se prefirió `max_depth=12` por ser más liviano y menos propenso a
overfitting. Las corridas de Optuna, pese a explorar un espacio más amplio de hiperparámetros
(`min_samples_split`, `min_samples_leaf`), no superaron el resultado del grid manual, lo que
sugiere que el modelo ya está cerca de su techo de performance con las features actuales.

In [ ]:
best_model = mlflow.sklearn.load_model(f"runs:/{best_run_id}/model")
print("Modelo cargado desde run:", best_run_id)

## 7. Registro del modelo en MLflow Model Registry

In [ ]:
mlflow.set_registry_uri("databricks-uc")

result = mlflow.register_model(
    model_uri=f"runs:/{best_run_id}/model",
    name=MODEL_NAME
)
print(result)

## 8. Validación end-to-end del modelo registrado

Databricks Free Edition no incluye Model Serving (funcionalidad paga), por lo que la validación
end-to-end del modelo registrado en Unity Catalog se realiza mediante inferencia local con
`mlflow.pyfunc.load_model`, cargando el modelo tal como lo consumiría un cliente real de un
endpoint de Model Serving.

In [ ]:
loaded_model = mlflow.pyfunc.load_model(f"runs:/{best_run_id}/model")
example = X_test.iloc[[0]]
pred = loaded_model.predict(example)
print("Input:", example.to_dict(orient="records")[0])
print("Predicción fare_amount:", pred)

## 9. Observabilidad y Drift con Evidently

Se compara `period == "reference"` (viajes de **2015-01**) contra `period == "current"`
(viajes de **2016**), usando el mejor modelo entrenado. Esto simula el escenario real de monitoreo:
¿cambió la distribución de los datos de entrada de un año a otro? ¿el modelo sigue funcionando bien?

In [ ]:
reference_pd = model_df.filter(F.col("period") == "reference").sample(fraction=0.05, seed=SEED).toPandas()
current_pd = model_df.filter(F.col("period") == "current").sample(fraction=0.05, seed=SEED).toPandas()

reference_pd["prediction"] = best_model.predict(reference_pd[FEATURE_COLS])
current_pd["prediction"] = best_model.predict(current_pd[FEATURE_COLS])

In [ ]:
from evidently import Report
from evidently.presets import DataDriftPreset, RegressionPreset
from evidently.future.datasets import Dataset, DataDefinition
from evidently import Regression

data_definition = DataDefinition(
    regression=[Regression(target="fare_amount", prediction="prediction")]
)

reference_dataset = Dataset.from_pandas(reference_pd, data_definition=data_definition)
current_dataset = Dataset.from_pandas(current_pd, data_definition=data_definition)

report = Report(metrics=[DataDriftPreset(), RegressionPreset()])
snapshot = report.run(current_data=current_dataset, reference_data=reference_dataset)
snapshot.save_html("/tmp/evidently_report.html")

displayHTML(open("/tmp/evidently_report.html").read())

### Explicación del monitoreo y drift

Comparando el período de referencia (viajes de enero 2015, ~4.850 filas en la muestra de
validación) contra el período actual (viajes de 2016, ~14.628 filas), Evidently marca drift
estadísticamente significativo en 7 de las 11 columnas analizadas (~64%): `trip_distance`,
`haversine_km`, `pickup_dayofweek`, `pickup_month`, `payment_type`, `trip_duration_min` y el
propio target `fare_amount`. No se detecta drift relevante en `passenger_count`, `pickup_hour`,
`RateCodeID` ni `VendorID`.

El drift en `pickup_month` es en gran parte un artefacto del diseño de muestreo: el período de
referencia contiene únicamente enero, mientras que el período actual mezcla enero/febrero/marzo
de 2016, por lo que no debe interpretarse como un cambio real de comportamiento sino como una
limitación de cómo se definieron los dos períodos.

En cuanto a la performance del modelo, el error creció al pasar de referencia a actual (RMSE
0.97 → 1.42 USD, MAE 0.28 → 0.30 USD), aunque el R² se mantiene alto en ambos casos
(0.990 → 0.981). Esto es consistente con el drift detectado en `trip_distance` / `trip_duration_min`
/ `fare_amount`: los viajes de 2016 tienen una distribución de distancias y tarifas levemente
distinta a la de enero 2015 (estacionalidad y evolución normal del tráfico/tarifas de la ciudad
entre años), lo que empeora algo la precisión del modelo sin comprometer su utilidad general.

## 10. Interpretabilidad con SHAP (opcional)

In [ ]:
import shap

sample_X = X_test.sample(min(500, len(X_test)), random_state=SEED)
explainer = shap.Explainer(best_model, X_train.sample(min(500, len(X_train)), random_state=SEED))
shap_values = explainer(sample_X)

shap.summary_plot(shap_values.values, sample_X, feature_names=FEATURE_COLS)

In [ ]:
import matplotlib.pyplot as plt

importances = best_model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]
plt.barh(np.array(FEATURE_COLS)[sorted_idx], importances[sorted_idx])
plt.xlabel("Feature Importance")
plt.title("Importancia de features - Random Forest (fare_amount)")
plt.gca().invert_yaxis()
plt.show()

## 11. Instrucciones para reproducir

El dataset `nyctaxi_sample.csv` está registrado como tabla `workspace.default.nyctaxi_sample`
en Unity Catalog (Databricks Free Edition). Las corridas de este notebook quedan registradas
en el experimento de MLflow `/Users/rsanchezfaris@itba.edu.ar/nuevoexperimento`, y el modelo
final se registra en Model Registry como `workspace.default.nyctaxi_fare_model`.

Todas las semillas aleatorias (muestreo, train/test split, Random Forest, Optuna) están fijadas
en `SEED = 584154` para que la ejecución sea reproducible. Los números reportados en las
secciones de justificación, drift y conclusiones se calcularon corriendo localmente el mismo
pipeline de limpieza/feature engineering (equivalente en pandas) y entrenamiento sobre
`tp/nyctaxi_sample.csv` con esa semilla; al ejecutar este notebook completo en Databricks con
PySpark sobre la misma tabla, los resultados deberían ser equivalentes.

## 12. Conclusiones

- **Mejor modelo:** `RandomForestRegressor(n_estimators=150, max_depth=12)`, seleccionado entre
  4 corridas del grid manual y 3 corridas adicionales de Optuna (7 en total), con RMSE=0.91 USD,
  MAE=0.29 USD y R²=0.991 sobre el conjunto de test.
- **Features más importantes** (según `feature_importances_`): `trip_distance` (~81%) y
  `trip_duration_min` (~16%) concentran ~97% de la importancia total, seguidas de lejos por
  `RateCodeID` (~2.5%). El resto de las features (Haversine, hora, día, mes, pasajeros, tipo
  de pago, vendor) aporta una influencia marginal. Esto es coherente con el dominio: la tarifa
  de un taxi en NYC depende centralmente de cuánto se viajó (distancia y tiempo), no de quién
  ni cuándo.
- **Drift 2015 vs 2016:** se detecta drift estadísticamente significativo en 7 de 11 columnas,
  con una degradación moderada del error del modelo (RMSE +46%, de 0.97 a 1.42 USD) al pasar
  del período de referencia al período actual, sin colapso del modelo (R² se mantiene por
  encima de 0.98).
- **Limitaciones:** la muestra usada (~400k filas) es una fracción del dataset completo
  (~47M filas); Databricks Free Edition no soporta Model Serving pago, por lo que la validación
  end-to-end se hizo con inferencia local en vez de un endpoint real; el período de referencia
  (solo enero 2015) introduce un sesgo estacional que infla artificialmente el drift detectado
  en `pickup_month`.